## Text2Mesh Colab Notebook
Colab notebook for [Text2Mesh](https://github.com/threedle/text2mesh). Notebook made by [mfrashad](https://github.com/mfrashad).

In [ ]:
import torch
import subprocess
import sys

# Check GPU availability
print("GPU available:", torch.cuda.is_available())

# If GPU exists, show device name
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Check CUDA version installed with PyTorch
print("PyTorch CUDA version:", torch.version.cuda)

# Check system-level CUDA (nvcc) if available
try:
    nvcc_version = subprocess.check_output(["nvcc", "--version"]).decode()
    print("\nSystem CUDA (nvcc):\n", nvcc_version)
except FileNotFoundError:
    print("\nSystem CUDA (nvcc) not installed.")

# Check Pytorch version
print("PyTorch version:", torch.__version__)

print("Python version:", sys.version)

: 

In [ ]:
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0

In [ ]:
#@title Setup
!pip install trimesh==3.9.33 einops==0.3.2 scipy==1.5.2 \
             siren-pytorch==0.1.5 usd-core==21.8 \
             torch==1.9.0 torchtext==0.10.0 torchvision==0.10.0 cython==0.29.20 \
             git+https://github.com/openai/CLIP.git@04f4dc2ca1ed0acc9893bd1a3b526a7e02c4bb10

In [ ]:
#@title Check kaolin version
import kaolin
print(f'Kaolin Version: {kaolin.__version__}')

In [ ]:
#@title Install Kaolin 0.15.0
!pip install kaolin==0.15.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.1.0_cu121.html

In [ ]:
#@title Trying later Kaolin version
!git clone --recursive https://github.com/NVIDIAGameWorks/kaolin
%cd kaolin
!python setup.py develop
%cd ..


In [ ]:
#@title Install dependency (Kaolin)
!git clone --recursive https://github.com/NVIDIAGameWorks/kaolin
%cd kaolin
!git checkout v0.10.0
!python setup.py develop
%cd ..

In [ ]:
#@title Get text2mesh
!git clone https://github.com/threedle/text2mesh
%cd text2mesh

In [ ]:
#@title Remesh low vertex objects
import ipywidgets as widgets

remesh_obj_path = widgets.Text(
    value='',
    description='Input OBJ:',
)

remesh_output_path = widgets.Text(
    value='./remeshed.obj',
    description='Output OBJ:',
)

display(remesh_obj_path, remesh_output_path)
# remesh_obj_path = "" #@param {type: "string"}
# remesh_output_path = "./remeshed.obj" #@param {type: "string"}

In [ ]:
!python remesh.py --obj_path {remesh_obj_path} \
                  --output_path {remesh_output_path}

In [ ]:
obj_path = "data/source_meshes/person.obj"  #@param {type: "string"}
n_iter = 750  #@param {type: "integer"}
prompt = "a 3D rendering of a ninja in unreal engine" #@param {type: "string"}
output_dir = "./results2"

In [ ]:
#@title Run text2mesh, the intermediate results can be seen in 'text2mesh/results' directory
!python main.py --run branch \
                --obj_path {obj_path} \
                --output_dir {output_dir} \
                --prompt "{prompt}" \
                --sigma 12.0  --clamp tanh --n_normaugs 4 --n_augs 1 --normmincrop 0.1 --normmaxcrop 0.4 \
                --geoloss --colordepth 2 --normdepth 2 --frontview --frontview_std 4 --clipavg view \
                --lr_decay 0.9 --clamp tanh --normclamp tanh  --maxcrop 1.0 --save_render --seed 29 \
                --n_iter {n_iter}  --learning_rate 0.0005 --normal_learning_rate 0.0005 --standardize --no_pe --symmetry --background 1 1 1

In [ ]:
#@title export the results
import matplotlib.pyplot as plt
import importlib
import PIL
importlib.reload(PIL.TiffTags)
import cv2
import os


frames = []
for i in range(0, n_iter, 100):
    img = cv2.imread(os.path.join(output_dir, f"iter_{i}.jpg"))
    frames.append(img)
    plt.figure(figsize=(20, 4))
    plt.axis("off")
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.show()

In [ ]:
#@title create video
from IPython.display import display, HTML
from base64 import b64encode
from tqdm.auto import tqdm
import cv2
fps = 2


video = cv2.VideoWriter("video.avi", 0, fps, frames[0].shape[:2][::-1]);
for im in tqdm(frames):
    video.write(im)
video.release()
!ffmpeg -y -i video.avi -pix_fmt yuv420p video.mp4 2> /dev/null
mp4 = open("video.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f"""
<video width={frames[0].shape[1]} controls>
      <source src="{data_url}" type="video/mp4">
</video>
"""))